# 03 — Rooflines and full-node scaling

**Question.** Where do measured bandwidth and compute limits meet, and how much throughput can both sockets sustain together?

The empirical envelope is `P(I) = min(P_compute, B × I)` and the knee is `P_compute / B`. Bandwidth and arithmetic intensity use explicit modeled bytes. Every envelope below matches the mode, host, and socket of its compute and bandwidth measurements. Only FP64 has an application-like intensity sweep.


In [ ]:
from pathlib import Path
import hashlib, json, sys
import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/hwbench').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from hwbench.analysis import (load_measurements, select_measurements, export_figure,
    memory_table, compute_table, locality_table, scaling_table, knees_table,
    theoretical_table, concurrent_table, bandwidth_scaling, cache_working_set,
    locality_figure, compute_scaling, fp64_rooflines, dual_scaling)
pio.renderers.default = 'notebook'
validation = json.loads((ROOT / 'data/processed/validation.json').read_text())
assert validation['status'] == 'passed', validation['errors']
assert validation['dataset_sha256'] == hashlib.sha256((ROOT / 'data/processed/measurements.csv').read_bytes()).hexdigest()
assert validation['selection_sha256'] == hashlib.sha256((ROOT / 'configs/analysis.json').read_bytes()).hexdigest()
df = select_measurements(load_measurements(ROOT / 'data/processed/measurements.csv'))
NOTEBOOK = '03_rooflines_and_scaling'
def table(frame):
    shown = frame.copy()
    if 'implementation' in shown:
        shown['implementation'] = shown['implementation'].replace({
            'native_amx_4_tiles': 'Native AMX tiles',
            'onednn_numactl_only_omp_affinity_unset': 'oneDNN GEMM',
            'native_avx512': 'AVX-512'})
    if 'metric' in shown:
        shown['metric'] = shown['metric'].replace({'bandwidth_gbs':'GB/s','performance_gops':'GFLOP/s','latency_ns':'ns/load'})
    shown = shown.rename(columns={'memory_mode':'Mode','socket':'Socket','memory_target':'Memory',
        'locality':'Placement','working_set_gib':'Working set (GiB)','implementation':'Compute path',
        'dtype':'Precision','m':'Matrix size','n_samples':'Samples','efficiency':'Scaling efficiency'})
    display(shown.round(3).style.set_uuid(NOTEBOOK).hide(axis='index'))


## 1. FP64: memory-bound and compute-bound regimes

**Conditions.** Socket 0, eight independent AVX-512 chains, nine intensity settings. Flat DDR/HBM use 8 GiB; Cache uses separate 16 and 96 GiB effective-memory cases. Lines are envelopes; open markers are measurements, with sample standard deviation. No Cache access is attributed to a specific tier.

In [ ]:
fig = fp64_rooflines(df, socket=0)
export_figure(fig, 'fp64_rooflines', NOTEBOOK)

In [ ]:
k=knees_table(df)
table(k[k.socket.eq(0)&k.dtype.eq('fp64')][['memory_mode','memory_target','working_set_gib','compute_gflops','bandwidth_gbs','knee_flop_per_byte']])

**Implication.** Below the knee, extra compute capability alone does not raise the modeled roof. Above it, bandwidth improvements alone do not raise the compute reference. The slope and plateau come from different kernels; their envelope is benchmark-specific and is not a strict counter-based bound for every application.

### FP32 and BF16 envelopes

The compact table below gives the same limits for FP32 and both BF16 implementations. No FP32/BF16 intensity measurements are invented. Native AMX and oneDNN remain distinct references.

In [ ]:
table(k[k.socket.eq(0)&k.dtype.isin(['fp32','bf16'])][['memory_mode','memory_target','working_set_gib','dtype','implementation','compute_gflops','bandwidth_gbs','knee_flop_per_byte']])

## 2. Measured two-socket scaling

**Conditions.** Each pair first completes initialization and warm-up, then synchronizes readiness before a shared clock window. Aggregate throughput is total completed work divided by the interval spanning both participants. At least 95% of the interval must overlap.

The denominator is the sum of socket 0 and socket 1 isolated medians from the same campaign, implementation, placement, and shape. A ratio of one means those isolated rates were retained simultaneously.

In [ ]:
fig = dual_scaling(df)
export_figure(fig, 'socket_scaling', NOTEBOOK)

In [ ]:
s=scaling_table(df)
table(s[['memory_mode','memory_target','dtype','implementation','dual_median','metric','efficiency']])

**Implication.** The absolute dual rate is the measured full-node capability for these independent workers. Efficiency reveals how simultaneous execution changes the isolated rates. This is not tensor-parallel application scaling and does not measure cross-socket communication performance. A ratio above one does not identify a causal mechanism.

## 3. Thesis evidence and remaining limits

Use memory results together with placement and working-set size; distinguish instruction throughput from achieved GEMM; use measured dual values when describing the full node. All figures and tables retain five-sample variability and matching configuration identity.

Unmeasured quantities include memory-controller traffic, HBM hit rates, loaded memory latency, power efficiency, and causes of residual performance gaps. The numerical tables exported to `data/processed/thesis/` are generated by the same functions used here.